# Analyze a PFM domain-wall run

For runs of `Run_PFM_DomainWalls.ipynb` (`MODALITY = "domain_wall_explorer"`): the walls the
explorer followed, the **PFM amplitude** (viridis) and **phase** (inferno) images at the commanded
and the measured (sensor) positions, where the walls are, the reference raster if the file has
one, and the open- vs closed-loop animations. Topography runs: `Analyze_SelfDriving_Scan.ipynb`.

## 1. Load a run

In [ ]:
import json, pathlib, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as _IPImage, Video as _IPVideo, display
from selfdriving_afm.config import LiveViewConfig
from selfdriving_afm.reconstruction import bin_image, inpaint_binned
from selfdriving_afm.viz import _KIND_STYLE, length_axis_ticks

SCAN_FILE = None   # None = the newest scan_domain_wall_explorer_*.npz in scan_data/, or a path

if SCAN_FILE is None:
    cands = sorted(p for p in pathlib.Path("scan_data").glob("scan_domain_wall_explorer_*.npz") if "_reanalyzed" not in p.name)
    if not cands:
        raise FileNotFoundError("no scan_domain_wall_explorer_*.npz in scan_data/ -- set SCAN_FILE")
    SCAN_FILE = str(cands[-1])
d = np.load(SCAN_FILE, allow_pickle=True)
KINDS = [str(k) for k in d["kinds"]]
OFFSETS = np.concatenate([[0], np.cumsum(np.asarray(d["segment_lengths"]))])
CHANNELS = [k[3:] for k in d.files if k.startswith("ch_")]
PFM = json.loads(str(d["pfm_json"])) if "pfm_json" in d.files else {}
HAS_SENSORS = "x_sensor" in d.files
CAL = None
if "sensor_calibration_json" in d.files:
    from selfdriving_afm.sensors import SensorCalibration
    CAL = SensorCalibration.from_dict(json.loads(str(d["sensor_calibration_json"])))
MODEL = json.loads(str(d["substrate_json"])) if "substrate_json" in d.files else {}
FEATURES = json.loads(str(d["features_json"])) if "features_json" in d.files else None
PRIORS = json.loads(str(d["priors_json"])) if "priors_json" in d.files else {}
UNITS = {k: PFM.get(k, {}).get("unit", "") for k in CHANNELS}
LABELS = {k: f"{PFM.get(k, {}).get('label', k)} ({UNITS[k]})" for k in CHANNELS}
CMAPS = {"amplitude": "viridis", "phase": "inferno"}

rate = float(d["sample_rate_hz"])
print(f"{SCAN_FILE}: {d['modality']}, {len(KINDS)} waveforms ({KINDS.count('explore')} explore, "
      f"{KINDS.count('connector')} moves), {OFFSETS[-1]:,} samples = {OFFSETS[-1] / rate:.1f} s of scanning at "
      f"{float(d['tip_velocity_V_s']):g} V/s")
for k in CHANNELS:
    c = PFM.get(k, {})
    print(f"  {LABELS[k]}: {c.get('source')} via {c.get('side')} loop {c.get('loop')} -> {c.get('bnc')} -> "
          f"{c.get('ai_channel')} (x {c.get('gain')})")
print(f"  'z' (live view / raster) = {str(d['z_label']) if 'z_label' in d.files else 'z'}; position sensors: {HAS_SENSORS}")
ENTERED = PRIORS.get("entered")
e = ENTERED
if PRIORS:
    print(f"  priors: domains {PRIORS['feature_min_V']:.3g}..{PRIORS['feature_max_V']:.3g} V, area half-width "
          f"{PRIORS['area_half_width_V']:.3g} V" + (f"  [entered: {e['feature_min_nm']:g}..{e['feature_max_nm']:g} nm, "
          f"half-width {e['area_half_width_nm']:g} nm]" if e else ""))
if MODEL:
    print(f"  walls: domains' amplitude {MODEL.get('amplitude_ref')}, phases {MODEL.get('phase_A_deg')} / "
          f"{MODEL.get('phase_B_deg')} deg")
WALLS = PFM.get("walls", {})
if FEATURES is None:
    print("  (this file has no features_json -- saved before it was recorded for this mode)")
else:
    print(f"  {len(FEATURES)} traced: " + ", ".join(
        f"{f['id']}{'(closed)' if f['closed'] else ''}" for f in FEATURES))

## 2. Settings
The image window is the run's (`recon_bounds`). Positions: `"measured"` (sensors, the real geometry)
or `"commanded"`.

In [ ]:
N_PIXELS = 200
BOUNDS = tuple(float(v) for v in d["recon_bounds"])
RECON_MODE = "linear"        # "linear" (sharp walls), "compressed_sensing", "fast", "gp"
POSITIONS = "measured" if HAS_SENSORS else "commanded"
_recon_cfg = LiveViewConfig(reconstruction=RECON_MODE, n_image_pixels=N_PIXELS)


def xy(positions=POSITIONS):
    if positions == "measured" and HAS_SENSORS:
        return np.asarray(d["x_sensor"]), np.asarray(d["y_sensor"])
    return np.asarray(d["x_cmd"]), np.asarray(d["y_cmd"])


def axes_units(ax, positions=POSITIONS, unit="um"):
    ax.set_xlim(BOUNDS[0], BOUNDS[1]); ax.set_ylim(BOUNDS[2], BOUNDS[3]); ax.set_aspect("equal")
    if positions == "measured" and CAL is not None:
        length_axis_ticks(ax, CAL, unit=unit)
        ax.set_xlabel(f"X ({unit}, sensor)"); ax.set_ylabel(f"Y ({unit}, sensor)")
    else:
        ax.set_xlabel("X (V)"); ax.set_ylabel("Y (V)")


def channel_range(name, img):
    if name == "phase":
        return -180.0, 180.0
    return tuple(np.nanpercentile(img, [1, 99]))

## 3. The path and the walls
The tip's path (measured positions), the traced wall crossings (each feature's outline, from
`features_json`) and every sample where the amplitude is below the wall level
(`amp_dip_fraction` x the domains' amplitude) -- the walls as the data shows them.

In [ ]:
X, Y = xy()
fig, ax = plt.subplots(figsize=(7.5, 7), layout="constrained")
for i in range(len(KINDS)):
    a, b = int(OFFSETS[i]), int(OFFSETS[i + 1])
    st = max(1, (b - a) // 1500)
    style = {k: v for k, v in _KIND_STYLE.get(KINDS[i], {}).items() if k != "label"}
    ax.plot(X[a:b:st], Y[a:b:st], **style)
if "amplitude" in CHANNELS and MODEL.get("amplitude_ref"):
    amp = np.asarray(d["ch_amplitude"])
    wall = amp < WALLS.get("amp_dip_fraction", 0.6) * MODEL["amplitude_ref"]
    st = max(1, int(wall.sum()) // 30000)
    ax.plot(X[wall][::st], Y[wall][::st], ".", ms=1, color="k", alpha=0.5, label="amplitude below the wall level")
if FEATURES:
    for f in FEATURES:
        o = np.asarray(f["outline"])
        if len(o):
            ax.plot(o[:, 0], o[:, 1], "o", ms=2, label=f"feature {f['id']}" + (" (closed)" if f["closed"] else ""))
axes_units(ax)
ax.legend(fontsize=7, loc="upper right", ncol=2)
ax.set_title("path, traced walls, low-amplitude samples")
plt.show()

## 4. Amplitude and phase images
Binned (the measured pixels) and reconstructed (`RECON_MODE`), for every channel the file has, at
the commanded and the measured positions. The phase is shown on its full circle (-180..180 deg).

In [ ]:
IMAGES = {}
pos_list = ["commanded", "measured"] if HAS_SENSORS else ["commanded"]
fig, axs = plt.subplots(len(pos_list), 2 * len(CHANNELS), figsize=(4.4 * 2 * len(CHANNELS), 4.2 * len(pos_list)),
                        squeeze=False, layout="constrained")
for r, pos in enumerate(pos_list):
    px, py = xy(pos)
    for k, name in enumerate(CHANNELS):
        v = np.asarray(d[f"ch_{name}"], dtype=float)
        ok = np.isfinite(v)
        B, _ = bin_image(np.column_stack([px[ok], py[ok]]), v[ok], N_PIXELS, BOUNDS)
        R = inpaint_binned(B, BOUNDS, _recon_cfg, mode=RECON_MODE)
        IMAGES[(pos, name)] = (B, R)
        lo, hi = channel_range(name, B)
        for j, (img, what) in enumerate(((B, "binned"), (R, RECON_MODE))):
            ax = axs[r, 2 * k + j]
            im = ax.imshow(img, origin="lower", extent=BOUNDS, cmap=CMAPS.get(name, "viridis"), vmin=lo, vmax=hi,
                           interpolation="nearest")
            fig.colorbar(im, ax=ax, label=LABELS[name], shrink=0.85)
            axes_units(ax, pos)
            ax.set_title(f"{name}, {what} -- {pos} positions", fontsize=9)
plt.show()

## 5. Where the walls are
For walls that run mostly in one direction (e.g. PPLN's straight walls): how many low-amplitude
samples fall at each X (and Y) -- the peaks are the walls. Positions in the image's units.

In [ ]:
if "amplitude" in CHANNELS and MODEL.get("amplitude_ref"):
    X, Y = xy()
    amp = np.asarray(d["ch_amplitude"])
    wall = amp < WALLS.get("amp_dip_fraction", 0.6) * MODEL["amplitude_ref"]
    BIN_V = 0.05          # histogram bins (V)
    SMOOTH_V = 0.1        # smoothing: a wall is ~a few tenths of a V wide (the dip below the wall level)
    MIN_SEP_V = 0.4       # two walls at least this far apart (a domain narrower than this: one peak)
    from scipy.ndimage import gaussian_filter1d
    from scipy.signal import find_peaks
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.2), layout="constrained")
    for ax, (v, lab) in zip(axs, ((X, "X"), (Y, "Y"))):
        edges = np.arange(BOUNDS[0] if lab == "X" else BOUNDS[2], (BOUNDS[1] if lab == "X" else BOUNDS[3]) + BIN_V, BIN_V)
        hw, _ = np.histogram(v[wall], bins=edges)
        ht, _ = np.histogram(v, bins=edges)
        c = 0.5 * (edges[1:] + edges[:-1])
        sm = SMOOTH_V / BIN_V
        frac = gaussian_filter1d(hw.astype(float), sm) / np.maximum(gaussian_filter1d(ht.astype(float), sm), 1)
        frac[ht == 0] = np.nan          # of the samples there, how many are on a wall
        ax.plot(c, frac, "-", lw=1)
        ax.set_xlabel(f"{lab} (V{', sensor frame' if HAS_SENSORS else ''})"); ax.set_ylabel("fraction on a wall")
        if lab == "X":
            pk, _ = find_peaks(np.nan_to_num(frac), height=0.2 * np.nanmax(frac), distance=max(1, int(MIN_SEP_V / BIN_V)))
            for p_ in pk:
                ax.axvline(c[p_], color="r", lw=0.6, alpha=0.6)
            um = [float(CAL.um(c[p_], 0)[0]) for p_ in pk] if CAL is not None else None
            print(f"{len(pk)} walls along X at", ", ".join(f"{c[p_]:+.2f} V" + (f" ({u:+.2f} um)" if um else "")
                                                         for p_, u in zip(pk, um or [None] * len(pk))))
            # where the explorer hardly went (a wall there would be missed): runs of barely visited bins
            low = ht < 0.02 * np.median(ht[ht > 0])
            runs, i = [], 0
            while i < len(low):
                if low[i]:
                    j = i
                    while j + 1 < len(low) and low[j + 1]:
                        j += 1
                    if (j - i + 1) * BIN_V >= 0.2:
                        runs.append((edges[i], edges[j + 1]))
                    i = j + 1
                else:
                    i += 1
            if runs:
                print("  barely visited along X (a wall there would be missed): "
                      + ", ".join(f"{a:+.2f}..{b:+.2f} V" for a, b in runs))
    plt.show()
else:
    print("no amplitude channel in this file")

## 6. Reference raster (if this run has one)
The raster recorded "z" only (the phase if it was on, else the amplitude), at the measured
positions when the sensors were on. Next to the self-driving run's image of the same signal.

In [ ]:
if "raster_binned_Z" in d.files:
    from selfdriving_afm import reference_raster
    RASTER = reference_raster.load(d, positions="auto")
    zname = "phase" if "phase" in str(d["z_label"]).lower() else "amplitude"
    rb = RASTER["bounds"]
    rec = inpaint_binned(RASTER["binned"], rb, _recon_cfg, mode="raster_columns")
    runB, runR = IMAGES.get(("measured" if RASTER["positions"] == "sensor" else "commanded", zname), (None, None))
    lo, hi = channel_range(zname, RASTER["binned"])
    panels = [(runR, BOUNDS, f"self-driving: {RECON_MODE}"), (reference_raster.for_display(RASTER["binned"]), rb,
              f"raster: binned ({int(d['raster_n_lines'])} lines)"), (rec, rb, "raster: column-wise PCHIP")]
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.6), layout="constrained")
    for ax, (img, b, t) in zip(axs, panels):
        if img is None:
            continue
        im = ax.imshow(img, origin="lower", extent=b, cmap=CMAPS.get(zname, "viridis"), vmin=lo, vmax=hi,
                       interpolation="nearest")
        axes_units(ax, "measured" if RASTER["positions"] == "sensor" else "commanded")
        ax.set_title(t, fontsize=9)
    fig.colorbar(im, ax=list(axs), label=LABELS.get(zname, zname), shrink=0.85)
    plt.show()
    print(f"raster: {int(d['raster_pixels']):,} pixels for the run's {int(d['raster_run_pixels']):,}; "
          f"{float(d['raster_scan_time_s']):.0f} s of scanning, {float(d['raster_wall_time_s']):.0f} s wall time")
else:
    print("no reference raster in this file")

# the raster's PFM signals (recorded with it when cfg.pfm was on): next to the run's, per signal
if "raster_channels" in d.files:
    rnames = [str(n) for n in d["raster_channels"]]
    rb = tuple(float(v) for v in d["raster_bounds"])
    where = "measured" if f"raster_ch_{rnames[0]}_binned_sensor" in d.files else "commanded"
    tag = "_sensor" if where == "measured" else ""
    fig, axs = plt.subplots(len(rnames), 3, figsize=(14, 4.4 * len(rnames)), squeeze=False, layout="constrained")
    for r, n in enumerate(rnames):
        B = np.asarray(d[f"raster_ch_{n}_binned{tag}"]); Rr = np.asarray(d[f"raster_ch_{n}_recon{tag}"])
        runR = IMAGES.get((where, n), (None, None))[1]
        lo, hi = channel_range(n, B)
        for ax, (img, bb, t) in zip(axs[r], ((runR, BOUNDS, f"self-driving {n}: {RECON_MODE}"),
                                             (reference_raster.for_display(B), rb, f"raster {n}: binned"),
                                             (Rr, rb, f"raster {n}: column-wise PCHIP"))):
            if img is None:
                ax.set_visible(False); continue
            im = ax.imshow(img, origin="lower", extent=bb, cmap=CMAPS.get(n, "viridis"), vmin=lo, vmax=hi,
                           interpolation="nearest")
            fig.colorbar(im, ax=ax, label=LABELS.get(n, n), shrink=0.85)
            axes_units(ax, where); ax.set_title(f"{t} ({where} positions)", fontsize=9)
    plt.show()

## 7. Animations: open vs closed loop, and closed loop in nm
`compare_animation` as in the topography analysis, per channel: rows = the self-driving run at the
commanded / measured positions (+ the raster's, for the signal it recorded). Saved next to the run.

In [ ]:
from selfdriving_afm import compare_animation

ANIM_CHANNELS = CHANNELS        # e.g. ["amplitude"]
ANIM_FORMAT = "gif"             # or "mp4" (needs ffmpeg)
ANIM_MAX_FRAMES = 60
ANIM_RECON_EVERY = 2
ANIM_DPI = 70
SHOW_ANIM = True
ANIMS = []
for name in ANIM_CHANNELS:
    stem = pathlib.Path(SCAN_FILE).stem
    for rows, unit, tag in ((("run_cmd", "run_sensor", "raster_cmd", "raster_sensor"), "um", "open_vs_closed"),
                            (("run_sensor", "raster_sensor"), "nm", "closed_loop_nm")):
        if tag == "closed_loop_nm" and not HAS_SENSORS:
            continue
        out = pathlib.Path(SCAN_FILE).with_name(f"{stem}_{name}_{tag}.{ANIM_FORMAT}")
        p = compare_animation.animate(
            d, out, rows=rows, bounds=BOUNDS, n_pixels=N_PIXELS, recon_cfg=_recon_cfg, recon_mode=RECON_MODE,
            max_frames=ANIM_MAX_FRAMES, dpi=ANIM_DPI, recon_every=ANIM_RECON_EVERY, length_unit=unit,
            cmap=CMAPS.get(name, "viridis"), channel=name, z_label=LABELS[name],
            unc_label=(f"{ENTERED['feature_min_nm']:g} nm" if ENTERED else None),
            title=f"{name}: {tag.replace('_', ' ')}")
        if p is not None:
            ANIMS.append(p)
if SHOW_ANIM:
    for p in ANIMS:
        display(_IPVideo(str(p), embed=True) if p.suffix == ".mp4" else _IPImage(filename=str(p)))